In [3]:
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")
os.environ["LANGSMITH_TRACING"]="true"

In [4]:
from langsmith import Client

client = Client()

# Create the dataset
dataset_name = "rag_dataset"
dataset = client.create_dataset(dataset_name=dataset_name)

# Create examples - all must have consistent "inputs" and "outputs" keys
client.create_examples(
    inputs=[
        {"question": "What is the capital of France?"},
        {"question": "What is LangChain?"},
        {"question": "What is LangSmith?"},
        {"question": "What is OpenAI?"},
        {"question": "What is Google?"},
        {"question": "What is Mistral?"},
    ],
    outputs=[
        {"answer": "Paris"},
        {"answer": "A framework for building LLM applications"},
        {"answer": "A platform for observing and evaluating LLM applications"},
        {"answer": "A company that creates Large Language Models"},
        {"answer": "A technology company known for search"},
        {"answer": "A company that creates Large Language Models"},
    ],
    dataset_id=dataset.id,
)

{'example_ids': ['36152eb8-89cb-4c82-9b74-67c423b22446',
  'db154b22-dcaf-4393-8efd-5405deaf9b8e',
  '34835e4d-b901-445f-8933-8f5d5e2f66f7',
  '27296e5f-6c39-4134-8aaa-2a36fff54314',
  '11801c03-1b01-4ab1-9f2c-367100f29086',
  'f030b024-7c7b-4511-9b66-02aad90e51b0'],
 'count': 6}

Define Metrics and (LLM AS JUDGE)



In [5]:
import os

from google import genai
from google.genai import types
from langsmith import wrappers, Client

from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")
os.environ["LANGSMITH_TRACING"]="true"

# Create Gemini Client (uses GOOGLE_API_KEY from environment)
gemini_client = genai.Client()

# Wrap Gemini Client for LangSmith tracing
google_client = wrappers.wrap_gemini(gemini_client)

# LangSmith Client
ls_client = Client()

eval_instructions = """
You are an expert professor specialized in grading students' answers.

Compare the student's answer with the reference answer.

Respond with ONLY one word:

CORRECT

or

INCORRECT
"""

def correctness(
    inputs: dict,
    outputs: dict,
    reference_outputs: dict,
) -> bool:
    prompt = f"""
Question:
{inputs['question']}

Reference Answer:
{reference_outputs['answer']}

Student Answer:
{outputs['response']}

{eval_instructions}
"""

    response = google_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0,
        ),
    )

    grade = response.text.strip().upper()

    return grade == "CORRECT"


#  +++++++++++++++++++ CONCISIONS++++++++++++



# Concisions-check wheather the actual output is less than 2x the length of expected output




def Concisions(outputs:dict,reference_outputs:dict)->bool:
     if "response" not in outputs:
         return False
     if "answer" not in reference_outputs:
         return False
     return int(len(outputs["response"])) <= 2 * int(len(reference_outputs["answer"]))

C:\Users\Asus\AppData\Local\Temp\ipykernel_25312\3717191614.py:18: LangSmithBetaWarning: Function wrap_gemini is in beta.
  google_client = wrappers.wrap_gemini(gemini_client)


NOW ITS time for run the rag system with evaluations

In [6]:
from google import genai
from google.genai import types
import os

google_client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))   

default_instruction="Respond to user questions in a short,concise manner (one sentence)"

def my_app(questions:str,model:str= "gemini-2.5-flash",instruction:str=default_instruction)->str:
    response=google_client.models.generate_content(
        model=model,
        contents=questions,
        config=types.GenerateContentConfig(
            system_instruction=instruction,
            temperature=0,
        )
    )
    return response.text


# call my_App for everydata point
def ls_target(input:dict)->dict:
    return {"response": my_app(input["question"])}

run the evaluation

In [7]:
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")


experiment_result=client.evaluate(
     ls_target,
     data=dataset_name,
     evaluators=[correctness, Concisions],
     experiment_prefix="rag-eval_genai"
)

C:\Users\Asus\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'rag-eval_genai-568d1ede' at:
https://smith.langchain.com/o/acda6f3a-0cfa-4353-910e-f27da82eba70/datasets/2894eace-a766-4afb-aae0-6ccc7587ba85/compare?selectedSessions=3480e909-0f11-4b4a-99c1-45f0804c211a




3it [00:13,  4.38s/it]Error running target function: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 2.943997165s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location'